Read local CRMLSListing*.csv and CRMLSSold*.csv files (202403-202604),
merge each type into one combined dataset, filter both to
PropertyType == 'Residential', and save the results as new CSVs.

# Install Package

In [1]:
import os
import re
import glob
import pandas as pd

# Read Files

In [2]:
DATA_DIR = r"D:\Meng\document\AU\Career\2026 Intern\IDX\2. Data Analyst summer 2026\csv"
 
START_MONTH = 202403
END_MONTH = 202604
 
LISTING_PATTERN = re.compile(r"^CRMLSListing(\d{6})\.csv$", re.IGNORECASE)
SOLD_PATTERN = re.compile(r"^CRMLSSold(\d{6})\.csv$", re.IGNORECASE)

listing_frames = {}  # {yyyymm: dataframe}, also exposed as list202403, list202404, ...
sold_frames = {}      # {yyyymm: dataframe}, also exposed as sold202403, sold202404, ...
 
for path in sorted(glob.glob(os.path.join(DATA_DIR, "CRMLSListing*.csv"))):
    m = LISTING_PATTERN.match(os.path.basename(path))
    if not m:
        continue
    yyyymm = int(m.group(1))
    if not (START_MONTH <= yyyymm <= END_MONTH):
        continue
    var_name = f"list{yyyymm}"
    df = pd.read_csv(path, low_memory=False)
    globals()[var_name] = df  # creates e.g. list202403 = pd.read_csv('CRMLSListing202403.csv')
    listing_frames[yyyymm] = df
    print(f"Read {var_name}: {df.shape[0]} rows, {df.shape[1]} columns")
 
for path in sorted(glob.glob(os.path.join(DATA_DIR, "CRMLSSold*.csv"))):
    m = SOLD_PATTERN.match(os.path.basename(path))
    if not m:
        continue
    yyyymm = int(m.group(1))
    if not (START_MONTH <= yyyymm <= END_MONTH):
        continue
    var_name = f"sold{yyyymm}"
    df = pd.read_csv(path, low_memory=False)
    globals()[var_name] = df  # creates e.g. sold202403 = pd.read_csv('CRMLSSold202403.csv')
    sold_frames[yyyymm] = df
    print(f"Read {var_name}: {df.shape[0]} rows, {df.shape[1]} columns")
 
if not listing_frames:
    raise SystemExit("No CRMLSListing files found in the 202403-202604 range. Check DATA_DIR.")
if not sold_frames:
    raise SystemExit("No CRMLSSold files found in the 202403-202604 range. Check DATA_DIR.")
 
# Flag any months missing from either series (not fatal, just a heads-up)
expected_months = []
y, m = 2024, 3
while (y, m) <= (2026, 4):
    expected_months.append(y * 100 + m)
    m += 1
    if m > 12:
        m = 1
        y += 1
 
missing_listing = sorted(set(expected_months) - set(listing_frames.keys()))
missing_sold = sorted(set(expected_months) - set(sold_frames.keys()))
if missing_listing:
    print(f"\nWarning: no CRMLSListing file found for months: {missing_listing}")
if missing_sold:
    print(f"Warning: no CRMLSSold file found for months: {missing_sold}")

Read list202403: 32282 rows, 84 columns
Read list202404: 36503 rows, 84 columns
Read list202405: 38796 rows, 82 columns
Read list202406: 35893 rows, 82 columns
Read list202407: 36340 rows, 82 columns
Read list202408: 35305 rows, 82 columns
Read list202409: 34625 rows, 82 columns
Read list202410: 34730 rows, 82 columns
Read list202411: 25128 rows, 82 columns
Read list202412: 19417 rows, 82 columns
Read list202501: 37469 rows, 82 columns
Read list202502: 33983 rows, 82 columns
Read list202503: 38492 rows, 82 columns
Read list202504: 40187 rows, 82 columns
Read list202505: 40271 rows, 82 columns
Read list202506: 26399 rows, 82 columns
Read list202507: 27345 rows, 82 columns
Read list202508: 25210 rows, 82 columns
Read list202509: 26923 rows, 82 columns
Read list202510: 27586 rows, 82 columns
Read list202511: 20677 rows, 82 columns
Read list202512: 18773 rows, 82 columns
Read list202601: 35302 rows, 82 columns
Read list202602: 32884 rows, 82 columns
Read list202603: 39153 rows, 82 columns


# Merging Check

In [3]:
def check_schema_consistency(frames_dict, label):
    """Compare column sets across all monthly files of one type before concatenating."""
    months = sorted(frames_dict.keys())
    reference_month = months[0]
    reference_cols = set(frames_dict[reference_month].columns)
 
    mismatches = {}
    for yyyymm in months:
        cols = set(frames_dict[yyyymm].columns)
        if cols != reference_cols:
            mismatches[yyyymm] = {
                "missing": reference_cols - cols,
                "extra": cols - reference_cols,
            }
 
    if mismatches:
        print(f"\n[{label}] Column mismatches found (reference = {reference_month}):")
        for yyyymm, diff in mismatches.items():
            print(f"  {yyyymm}: missing={diff['missing']}, extra={diff['extra']}")
        print(f"[{label}] Will merge with an outer join on columns "
              f"(mismatched columns filled with NaN).")
    else:
        print(f"\n[{label}] All {len(months)} files have matching columns. Safe to merge.")
 
    return mismatches
 
 
listing_mismatches = check_schema_consistency(listing_frames, "Listing")
sold_mismatches = check_schema_consistency(sold_frames, "Sold")


[Listing] Column mismatches found (reference = 202403):
  202405: missing={'BuyerAgencyCompensation', 'BuyerAgencyCompensationType'}, extra=set()
  202406: missing={'BuyerAgencyCompensation', 'BuyerAgencyCompensationType'}, extra=set()
  202407: missing={'BuyerAgencyCompensation', 'BuyerAgencyCompensationType'}, extra=set()
  202408: missing={'BuyerAgencyCompensation', 'BuyerAgencyCompensationType'}, extra=set()
  202409: missing={'BuyerAgencyCompensation', 'BuyerAgencyCompensationType'}, extra=set()
  202410: missing={'BuyerAgencyCompensation', 'BuyerAgencyCompensationType'}, extra=set()
  202411: missing={'BuyerAgencyCompensation', 'BuyerAgencyCompensationType'}, extra=set()
  202412: missing={'BuyerAgencyCompensation', 'BuyerAgencyCompensationType'}, extra=set()
  202501: missing={'BuyerAgencyCompensation', 'BuyerAgencyCompensationType'}, extra=set()
  202502: missing={'BuyerAgencyCompensation', 'BuyerAgencyCompensationType'}, extra=set()
  202503: missing={'BuyerAgencyCompensation

# Merge Files into Listing and Sold

In [4]:
# Row counts before concatenation (sum of each individual monthly file)
listing_rows_before_concat = sum(df.shape[0] for df in listing_frames.values())
sold_rows_before_concat = sum(df.shape[0] for df in sold_frames.values())
print(f"\nListing rows before concatenation: {listing_rows_before_concat}")
print(f"Sold rows before concatenation: {sold_rows_before_concat}")
 
Listing = pd.concat(
    [listing_frames[mth] for mth in sorted(listing_frames.keys())],
    ignore_index=True, sort=False
)
Sold = pd.concat(
    [sold_frames[mth] for mth in sorted(sold_frames.keys())],
    ignore_index=True, sort=False
)
 
# Row counts after concatenation
print(f"Listing rows after concatenation: {Listing.shape[0]}")
print(f"Sold rows after concatenation: {Sold.shape[0]}")
 
assert Listing.shape[0] == listing_rows_before_concat, "Row count mismatch after Listing concat!"
assert Sold.shape[0] == sold_rows_before_concat, "Row count mismatch after Sold concat!"


Listing rows before concatenation: 838693
Sold rows before concatenation: 577824
Listing rows after concatenation: 838693
Sold rows after concatenation: 577824


In [5]:
# All columns are kept (not dropped) even when a given month's file didn't have them
# pd.concat(sort=False) NaN-fills those rows automatically
# Report completeness for any column that wasn't present in every monthly file, 
# so sparse columns are visible before they're used in analysis.
def report_inconsistent_column_completeness(frames_dict, merged_df, label):
    months = sorted(frames_dict.keys())
    reference_cols = set(frames_dict[months[0]].columns)
    all_cols = set(merged_df.columns)
    inconsistent_cols = sorted(all_cols.symmetric_difference(reference_cols) | (
        {c for c in all_cols if any(c not in frames_dict[mth].columns for mth in months)}
    ))
    if not inconsistent_cols:
        return
    print(f"\n[{label}] Completeness for columns not present in every monthly file:")
    for col in inconsistent_cols:
        non_null = merged_df[col].notna().sum()
        pct = 100 * non_null / len(merged_df)
        print(f"  {col}: {non_null}/{len(merged_df)} non-null ({pct:.1f}%)")
 
 
report_inconsistent_column_completeness(listing_frames, Listing, "Listing")
report_inconsistent_column_completeness(sold_frames, Sold, "Sold")


[Listing] Completeness for columns not present in every monthly file:
  BuyerAgencyCompensation: 68689/838693 non-null (8.2%)
  BuyerAgencyCompensationType: 68699/838693 non-null (8.2%)

[Sold] Completeness for columns not present in every monthly file:
  BuyerAgencyCompensation: 47874/577824 non-null (8.3%)
  BuyerAgencyCompensationType: 47854/577824 non-null (8.3%)
  BuyerAgentAOR: 524077/577824 non-null (90.7%)
  ListAgentAOR: 529699/577824 non-null (91.7%)
  OriginatingSystemName: 66643/577824 non-null (11.5%)
  OriginatingSystemSubName: 66643/577824 non-null (11.5%)
  latfilled: 143709/577824 non-null (24.9%)
  lonfilled: 143709/577824 non-null (24.9%)


# Filter PropertyType == 'Residential' Only

In [6]:
listing_rows_before_filter = Listing.shape[0]
sold_rows_before_filter = Sold.shape[0]
 
Listing_Residential = Listing[Listing["PropertyType"] == "Residential"].copy()
Sold_Residential = Sold[Sold["PropertyType"] == "Residential"].copy()
 
listing_rows_after_filter = Listing_Residential.shape[0]
sold_rows_after_filter = Sold_Residential.shape[0]
 
print(f"\nListing rows before Residential filter: {listing_rows_before_filter}")
print(f"Listing rows after Residential filter: {listing_rows_after_filter}")
print(f"Sold rows before Residential filter: {sold_rows_before_filter}")
print(f"Sold rows after Residential filter: {sold_rows_after_filter}")
 
if listing_rows_after_filter == 0 or sold_rows_after_filter == 0:
    print("\nWarning: filter produced 0 rows for one or both datasets. "
          "Check actual values with Listing['PropertyType'].unique() / "
          "Sold['PropertyType'].unique() in case the label differs "
          "(e.g. 'Residential ' with trailing space, or different casing).")
 
listing_output_path = os.path.join(DATA_DIR, "Listing_Residential_202403_202604.csv")
sold_output_path = os.path.join(DATA_DIR, "Sold_Residential_202403_202604.csv")
 
Listing_Residential.to_csv(listing_output_path, index=False)
Sold_Residential.to_csv(sold_output_path, index=False)
 
print(f"\nSaved: {listing_output_path}")
print(f"Saved: {sold_output_path}")


Listing rows before Residential filter: 838693
Listing rows after Residential filter: 533052
Sold rows before Residential filter: 577824
Sold rows after Residential filter: 389797

Saved: D:\Meng\document\AU\Career\2026 Intern\IDX\2. Data Analyst summer 2026\csv\Listing_Residential_202403_202604.csv
Saved: D:\Meng\document\AU\Career\2026 Intern\IDX\2. Data Analyst summer 2026\csv\Sold_Residential_202403_202604.csv


# Row Counts Comparison With Filtering

In [7]:
summary = pd.DataFrame({
    "Dataset": ["Listing", "Sold"],
    "Rows_Before_Concat": [listing_rows_before_concat, sold_rows_before_concat],
    "Rows_After_Concat": [Listing.shape[0], Sold.shape[0]],
    "Rows_Before_Residential_Filter": [listing_rows_before_filter, sold_rows_before_filter],
    "Rows_After_Residential_Filter": [listing_rows_after_filter, sold_rows_after_filter],
})
 
print("\nRow Count Summary:")
print(summary.to_string(index=False))


Row Count Summary:
Dataset  Rows_Before_Concat  Rows_After_Concat  Rows_Before_Residential_Filter  Rows_After_Residential_Filter
Listing              838693             838693                          838693                         533052
   Sold              577824             577824                          577824                         389797
